# 📊 Dataset Information

**Note:** The dataset for this project is **not included in this repository** to keep the repository lightweight.  

You can download the dataset from **Kaggle** using the link below:  

[Download The Nepal Flood Weather Dataset](https://www.kaggle.com/datasets/tejal5kunjir/nepal-flood-and-weather-dataset-2023-2026)  

Please make sure to place the downloaded CSV file in the same directory as this notebook before running any code.

# Importing

## Import Library

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

ModuleNotFoundError: No module named 'numpy'

## Import CSV And convert to DataFrame

In [ ]:
df = pd.read_csv('nepal_flood_weather_dataset_kaggle_2023_2026.csv')

# Preprocessing

## Frist five row

In [ ]:
df.head()

## last Five row

In [ ]:
df.tail()

## Shape of our dataset

In [ ]:
df.shape

## List out all columns

In [ ]:
df.columns

## Datatype of each columns

In [ ]:
df.dtypes

## Information of all Columns

In [ ]:
df.info()

## Check Null Value

In [ ]:
df.isnull().sum()

## Check Dupicate Value

In [ ]:
df.duplicated().sum()

## Summary

In [ ]:
df.describe()

# EDA

In [ ]:
def show_fig():
    plt.tight_layout()
    plt.show()

plot_no = 1

In [ ]:
fig = plt.figure(figsize=(10,6))
sns.histplot(data=df, x='temperature_mean_c', kde=True)
plt.title(f'{plot_no}. Mean Temperature Distribution Across the Dataset')
show_fig()
plot_no += 1

In [ ]:
fig = plt.figure(figsize=(10,6))
sns.scatterplot(data=df, x='precipitation_mm', y='river_discharge_m3s', alpha=0.5)
plt.title(f'{plot_no}. Precipitation vs River Discharge Relationship')
show_fig()
plot_no += 1

In [ ]:
fig = plt.figure(figsize=(10,6))
sns.regplot(data=df, x='rain_mm', y='river_discharge_m3s', scatter_kws={'alpha':0.3}, line_kws={'color':'red'})
plt.title(f'{plot_no}. Rainfall–River Discharge Trend and Flow Response')
show_fig()
plot_no += 1

In [ ]:
fig = plt.figure(figsize=(10,6))
sns.scatterplot(data=df, x='soil_moisture_0_100cm_m3m3', y='river_discharge_m3s', hue='basin', alpha=0.7)
plt.title(f'{plot_no}. Soil Moisture vs River Discharge Across Basins')
show_fig()
plot_no += 1

In [ ]:
fig = plt.figure(figsize=(10,6))
sns.boxplot(data=df, x='basin', y='precipitation_mm')
plt.xticks(rotation=45)
plt.title(f'{plot_no}. Rainfall Variability and Extreme Precipitation by Basin')
show_fig()
plot_no += 1

In [ ]:
fig = plt.figure(figsize=(10,6))
sns.boxplot(data=df, x='basin', y='temperature_mean_c')
plt.xticks(rotation=45)
plt.title(f'{plot_no}. Temperature Distribution Differences Across Basins')
show_fig()
plot_no += 1

In [ ]:
fig = plt.figure(figsize=(10,6))
sns.scatterplot(data=df, x='temperature_mean_c', y='dew_point_mean_c', hue='relative_humidity_mean_pct', palette='viridis', alpha=0.7)
plt.title(f'{plot_no}. Temperature–Dew Point Relationship Under Varying Humidity')
show_fig()
plot_no += 1

In [ ]:
fig = plt.figure(figsize=(10,6))
sns.scatterplot(data=df, x='relative_humidity_mean_pct', y='precipitation_mm', hue='basin', alpha=0.7)
plt.title(f'{plot_no}. Relative Humidity vs Precipitation Across Basins')
show_fig()
plot_no += 1

In [ ]:
fig = plt.figure(figsize=(10,6))
sns.scatterplot(data=df, x='wind_speed_max_kmh', y='river_discharge_m3s', hue='precipitation_mm', palette='coolwarm', alpha=0.7)
plt.title(f'{plot_no}. Wind Speed vs River Discharge Under Different Rainfall Conditions')
show_fig()
plot_no += 1

In [ ]:
fig = plt.figure(figsize=(10,6))
sns.scatterplot(data=df, x='wind_gusts_max_kmh', y='precipitation_mm', hue='basin', alpha=0.7)
plt.title(f'{plot_no}. Extreme Wind Gusts vs Precipitation Across Basins')
show_fig()
plot_no += 1

In [ ]:
fig = plt.figure(figsize=(10,6))
sns.barplot(data=df, x='basin', y='river_discharge_m3s', estimator='mean', errorbar=None)
plt.xticks(rotation=45)
plt.title(f'{plot_no}. Average River Discharge Comparison Across Basins')
show_fig()
plot_no += 1

In [ ]:
fig = plt.figure(figsize=(10,6))
sns.barplot(data=df, x='basin', y='soil_moisture_0_100cm_m3m3', estimator='mean', errorbar=None)
plt.xticks(rotation=45)
plt.title(f'{plot_no}. Average Soil Moisture Comparison Across Basins')
show_fig()
plot_no += 1

In [ ]:
fig = plt.figure(figsize=(10,6))
sns.lineplot(data=df, x='date', y='precipitation_mm', hue='basin', estimator='mean', errorbar=None)
plt.xticks(rotation=45)
plt.title(f'{plot_no}. Temporal Rainfall Trends Across River Basins')
show_fig()
plot_no += 1

In [ ]:
fig = plt.figure(figsize=(10,6))
sns.scatterplot(data=df, x='elevation_m', y='river_discharge_m3s', hue='basin', alpha=0.7)
plt.title(f'{plot_no}. Elevation vs River Discharge Across Basins')
show_fig()
plot_no += 1

In [ ]:
fig = plt.figure(figsize=(12,8))
corr = df.select_dtypes(include='number').corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title(f'{plot_no}. Correlation Heatmap of Hydrological and Weather Variables')
show_fig()
plot_no += 1

# Model Training

## Prepare features and target

In [ ]:
target = 'river_discharge_m3s'

X = df.drop(columns=[target, 'date'], errors='ignore')
y = pd.to_numeric(df[target], errors='coerce')

## Remove invalid target values

In [ ]:
valid = y.notna() & np.isfinite(y)
X = X.loc[valid]
y = y.loc[valid]

## Identify numerical and categorical columns

In [ ]:
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

## Handle missing values and encode categories

In [ ]:
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

## Combine preprocessing steps

In [ ]:
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
], remainder='drop')

## Split data into training and testing sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Build a supervised Extra Trees regression model

In [ ]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', ExtraTreesRegressor(
        n_estimators=150,
        min_samples_leaf=2,
        max_features=0.9,
        random_state=42,
        n_jobs=-1
    ))
])

## Train the model

In [ ]:
model.fit(X_train, y_train)

## Predict river discharge

In [ ]:
y_pred = model.predict(X_test)

## Evaluate model performance

In [ ]:
print(f"R² Accuracy Score: {r2_score(y_test, y_pred):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

## Plot actual vs predicted river discharge

In [ ]:
fig = plt.figure(figsize=(10,6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.6)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--')
plt.xlabel('Actual River Discharge (m³/s)')
plt.ylabel('Predicted River Discharge (m³/s)')
plt.title('Actual vs Predicted River Discharge')
show_fig()